In [1]:
import configparser
from sklearn.model_selection import ParameterGrid
import os
import numpy as np

In [ ]:
def create_directory(folderpath):

    os.makedirs(folderpath)

    os.system('cp /nfs/scistore26/saricgrp/dpersoon/vesicles/code/*.py' + ' ' + folderpath)



In [3]:
def create_config(folderpath, base_params):
    cfg = configparser.ConfigParser()
    cfg.optionxform = str
    cfg['_'] = {}
    
    for key in base_params:
        cfg['_'][key] = str(base_params[key])

    f=open(folderpath + '/cfg.ini', 'w', encoding='utf-8')
    cfg.write(f)

In [ ]:
def write_slurm(folderpath, name, string):
    f = open('%s/%s.slurm'%(folderpath,name), 'w')
    f.write('''#!/bin/bash -l
#
#SBATCH --job-name=%s
#SBATCH --output=%s.out
#
#
#SBATCH --ntasks-per-core=1
#
#Define the number of nodes the job should be distributed on
#SBATCH --nodes=1
#
#Total number of CPU cores to be used for the MPI job
#SBATCH --ntasks=4
#
#Define the number of hours the job should run. 
#Maximum runtime is limited to 10 days, ie. 240 hours
#SBATCH --time=60:00:00
#
#Define the amount of RAM used per CPU in GigaBytes
#In distributed memory applications the total amount of RAM 
#used will be:   number of CPUs * memory per CPU
#SBATCH --mem-per-cpu=10G
#
#Pick whether you prefer requeue or not. If you use the --requeue
#option, the requeued job script will start from the beginning, 
#potentially overwriting your previous progress, so be careful.
#For some people the --requeue option might be desired if their
#application will continue from the last state.
#Do not requeue the job in the case it fails.
#SBATCH --no-requeue

# 
##SBATCH --exclude=lemeshko07,bjoern22,bigterra152,lemeshko04,bjoern53,bjoern51,bea82
#for pure MPI jobs the number of threads has to be one
export OMP_NUM_THREADS=1
#
#
#run the respective binary through SLURM's srun
            
module load openmpi/4.1.6
module load netcdf/4.9.2
module load pnetcdf/1.12.3

micromamba activate /nfs/scistore26/saricgrp/dpersoon/micromamba/envs/vesicles
            
mpirun -np 4 /nfs/scistore26/saricgrp/dpersoon/micromamba/envs/vesicles/bin/lmp -in %s/%s
''' % (name, name, folderpath, string))
    f.close()

In [ ]:
all_params = {'fraction_negative_curve': [0.1, 0.2, 0.4, 0.6],
              'reduced_head_diameter': [0.5, 0.6, 0.7, 0.8],
              'seed': 1,
              'temperature':1.1,
              'pressure': 0,
              'N_mebrane_lipids_A': 4000,
              'R_vesicle_A': 20,
              'N_membrane_lipids_B': 4000,
              'R_vesicle_B': 20,
              'R_intervesicle': 16}

multiple_values = []

for key in all_params:
    if len(all_params[key]) > 1:
        multiple_values.append(key)

param_grid = ParameterGrid(all_params)
for i, params in enumerate(param_grid):

    extra_string = ''
    for s in multiple_values:
        if s=='seed':
            extra_string += '%s_%d/'%(s, params[s])
        if s=='fraction_negative_curve':
            extra_string += '%s_%.2f/'%('x', params[s])
        if s=='reduced_head_diameter':
            extra_string += '%s_%.2f/'%('rhd', params[s])
        else:
            extra_string += '%s_%.4f/'%(s, params[s])       
    path = '/nfs/scistore26/saricgrp/dpersoon/vesicles/data/'
    name = 'two_vesicles_force'
    folderpath = path + name
    full_path = folderpath + '/' + extra_string

    create_directory(full_path)
    create_config(full_path, params)
    os.chdir(full_path)
    os.system('python MultiVesicleData.py')
    os.system('python VesicleInput.py')
    write_slurm(full_path, name, 'in.local')
    os.system('sbatch %s.slurm'%(name))


Submitted batch job 57619314
Submitted batch job 57619315
Submitted batch job 57619316
Submitted batch job 57619317
Submitted batch job 57619318
Submitted batch job 57619319
Submitted batch job 57619320
Submitted batch job 57619321
Submitted batch job 57619322
Submitted batch job 57619323
Submitted batch job 57619324
Submitted batch job 57619325
Submitted batch job 57619326
Submitted batch job 57619327
Submitted batch job 57619328
Submitted batch job 57619329
Submitted batch job 57619330
Submitted batch job 57619331
Submitted batch job 57619332
Submitted batch job 57619333
Submitted batch job 57619334
Submitted batch job 57619335
Submitted batch job 57619336
Submitted batch job 57619337
Submitted batch job 57619338
Submitted batch job 57619339
Submitted batch job 57619340
Submitted batch job 57619341
Submitted batch job 57619342
Submitted batch job 57619343
Submitted batch job 57619344
